# Fig. 4 encoder redesign with the corrected direct Reynolds projection set

This notebook regenerates a Fig. 4-style encoder diagnostic **without overwriting the paper figure**. It is meant to be run after we corrected the projection-set logic for the orientation embedding.

The current paper figure is included as `figs/figure_encoder.pdf` in [`02_results_new.tex`](/data/home/umang/Materials/Reynolds-QSR_paper/Paper/EBSD_SR_Nature_v3/sections/02_results_new.tex:65). In that caption, panels **a,b** are the scalar fields

$$
s(\mathbf n)=\left\| f\!\left(R(\mathbf n,60^\circ)\right)-f(I)\right\|
$$

for FCC/$O$ and HCP/$D_6$, respectively. So the important audit here is: **does panel b change when HCP uses the direct Reynolds projector basis instead of the old Cartesian tensor-orbit recipe?**

Short answer: yes. The HCP representation changes from the legacy tensor route

$$
2\times 2e + 1\times 4e + 1\times 6e \quad (32\text{ dimensions})
$$

to the corrected direct Reynolds projector route

$$
1\times 2e + 1\times 4e + 2\times 6e \quad (40\text{ dimensions}).
$$

FCC through $l\le 4$ remains

$$
1\times 4e \quad (9\text{ dimensions}).
$$

All orientations here are treated in the **active convention**, consistent with the methods text: $q$ acts as $R(q)v$, and right multiplication by crystal symmetry $q\otimes S$ is the quotient action.

For the stochastic-looking diagnostics, the helper now uses deterministic low-discrepancy sampling: Hopf/Fibonacci quaternions on $S^3$ for orientation-pair and symmetry-spread samples, and Fibonacci directions on $S^2$ for the local 0--5$^\circ$ inset. This removes random clumping while preserving the full active/right-symmetry convention.

## Code path being exercised

This notebook intentionally keeps the numerical implementation in [`fig4_encoder_correct_projection_redesign.py`](/data/home/umang/Materials/Reynolds-QSR_paper/analysis/fig4_encoder_correct_projection_redesign.py:1), so the notebook stays readable and rerunnable.

The helper calls the actual model embedding code:

- The direct projector basis is built as $P_l=|G|^{-1}\sum_g D^l(g)$ in [`local_iso_embedding.py`](/data/home/umang/Materials/Reynolds-QSR_paper/models/local_iso_embedding.py:368).
- The old tensor-product HCP path is still available for comparison in [`local_iso_embedding.py`](/data/home/umang/Materials/Reynolds-QSR_paper/models/local_iso_embedding.py:636).
- The corrected direct-Reynolds route is selected by `embedding_mode="direct_reynolds"` and keeps the projector rank itself in [`local_iso_embedding.py`](/data/home/umang/Materials/Reynolds-QSR_paper/models/local_iso_embedding.py:753).
- The helper's active axis-angle and scalar-first quaternion operations are in [`fig4_encoder_correct_projection_redesign.py`](/data/home/umang/Materials/Reynolds-QSR_paper/analysis/fig4_encoder_correct_projection_redesign.py:88).

The notebook produces two main outputs:

1. A redesigned Fig. 4 candidate using corrected FCC/O and HCP/D6 direct-Reynolds embeddings.
2. A focused audit showing legacy HCP panel b vs corrected HCP panel b, normalized to expose the shape change rather than just scale.

In [ ]:
from pathlib import Path
import json
import sys

from IPython.display import Image, display

REPO_ROOT = Path("/data/home/umang/Materials/Reynolds-QSR_paper")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.fig4_encoder_correct_projection_redesign import DEFAULT_OUTPUT_DIR, run_all

print(f"repo root: {REPO_ROOT}")
print(f"output dir: {DEFAULT_OUTPUT_DIR}")

## Run settings

Use `QUICK_RUN=True` for a fast smoke test. Use `QUICK_RUN=False` for the normal rerun that writes the figure assets. The defaults are CPU-safe; the computation is mostly small Wigner-D embedding evaluation and plotting.

In [ ]:
QUICK_RUN = False

if QUICK_RUN:
    N_THETA = 31
    N_PHI = 61
    N_PAIRS = 512
    N_SPREAD = 128
else:
    N_THETA = 181
    N_PHI = 361
    N_PAIRS = 24000
    N_SPREAD = 24000

CHUNK_SIZE = 8192
OUTPUT_DIR = DEFAULT_OUTPUT_DIR

dict(
    quick_run=QUICK_RUN,
    n_theta=N_THETA,
    n_phi=N_PHI,
    n_pairs=N_PAIRS,
    n_spread=N_SPREAD,
    chunk_size=CHUNK_SIZE,
    output_dir=str(OUTPUT_DIR),
)

## Regenerate the corrected fields, curves, and figures

This cell constructs three embeddings:

- corrected FCC/O direct Reynolds, $l\le 4$;
- corrected HCP/D6 direct Reynolds, $l\le 6$;
- legacy HCP/D6 tensor-product route, used only as an audit reference.

For each corrected embedding, the helper also computes a finite-difference local metric scale at identity. This is only for plotting/calibration of distances; it does **not** change the embedding basis itself.

In [ ]:
result = run_all(
    OUTPUT_DIR,
    n_theta=N_THETA,
    n_phi=N_PHI,
    n_pairs=N_PAIRS,
    n_spread=N_SPREAD,
    chunk_size=CHUNK_SIZE,
)

{k: v for k, v in result.items() if k != "summaries"}

## Numerical summary

Check the emitted irreps first. The key expected values are:

- FCC corrected: `1x4e`, dimension 9.
- HCP corrected: `1x2e+1x4e+2x6e`, dimension 40.
- HCP legacy: `2x2e+1x4e+1x6e`, dimension 32.

The final `hcp_legacy_vs_corrected` block quantifies how much Fig. 4b changes after z-score normalization, so it is comparing the **shape of the scalar field**, not merely its absolute scale.

In [ ]:
summary = json.loads(Path(result["summary_json"]).read_text())
summary

## Redesigned Fig. 4 candidate

This figure uses the corrected direct-Reynolds projection set for both crystal systems. Panel b is therefore the updated HCP/D6 result. The PDF emitted by this cell is the source for the paper-integrated `figure_encoder_corrected_projection.pdf` asset.

In [ ]:
display(Image(filename=result["fig4_png"]))

## Focused HCP panel-b audit

This is the surgical comparison: old HCP tensor-route scalar field versus corrected HCP direct-Reynolds scalar field. Both are z-scored before differencing, because the direct projector path currently uses orthonormal projector bases and a local plotting calibration rather than the old hand-tuned tensor normalizers.

In [ ]:
display(Image(filename=result["hcp_audit_png"]))

## Saved files

The notebook writes a PNG/PDF pair for the redesigned figure, a PNG/PDF pair for the HCP audit, plus compressed numerical arrays and a JSON summary.

In [ ]:
for key in ["fig4_png", "fig4_pdf", "hcp_audit_png", "hcp_audit_pdf", "summary_json", "field_npz"]:
    path = Path(result[key])
    print(f"{key:>14}: {path}  exists={path.exists()}  size={path.stat().st_size if path.exists() else 'missing'}")

## Optional paper integration

This notebook deliberately does **not** overwrite the old [`Paper/EBSD_SR_Nature_v3/figs/figure_encoder.pdf`](/data/home/umang/Materials/Reynolds-QSR_paper/Paper/EBSD_SR_Nature_v3/figs/figure_encoder.pdf). The paper should include a separate corrected asset named `figure_encoder_corrected_projection.pdf`, so the old figure remains available for comparison.

In [ ]:
COPY_TO_PAPER = False

if COPY_TO_PAPER:
    import shutil

    paper_fig = REPO_ROOT / "Paper" / "EBSD_SR_Nature_v3" / "figs" / "figure_encoder_corrected_projection.pdf"
    shutil.copy2(result["fig4_pdf"], paper_fig)
    print(f"copied to {paper_fig}")
else:
    print("COPY_TO_PAPER is False; paper figure left unchanged.")